In [ ]:
import time
import cv2
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from ops.models import TSN
from archs.mobilenet_v3_deptheca_mega import MobileNetV3_DepthECAmega

# === Setup ===
ckpt_path = 'best.pth.tar'
label_path = 'datas/jester/category.txt'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_segments = 8
use_fp16 = False  # Disable to avoid display issues
frame_queue = []

# === Load labels ===
labels = [l.strip() for l in open(label_path, encoding='utf-8') if l.strip()]

# === Load model ===
model = TSN(len(labels), num_segments=num_segments,
            base_model='mobilenetv3_deptheca_mega',
            dropout=0.0, partial_bn=True, is_shift=True)

ckpt = torch.load(ckpt_path, map_location=device)
state_dict = ckpt.get('state_dict', ckpt)
clean_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
model.load_state_dict(clean_dict, strict=False)

model.to(device).eval()

# === Image transform ===
transform = T.Compose([
    T.Resize(256), T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
])

# === Open camera ===
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

if not cap.isOpened():
    raise RuntimeError("❌ Cannot open webcam")

cv2.namedWindow("🖐️ Gesture Recognition", cv2.WINDOW_NORMAL)
cv2.resizeWindow("🖐️ Gesture Recognition", 1280, 720)

print("🎥 Webcam feed started. Press 'q' to quit.")

ema = None
prev = time.time()

# === Inference loop ===
while True:
    ret, frame = cap.read()
    if not ret:
        print("⚠️ Failed to grab frame.")
        continue

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb)
    tensor = transform(img).unsqueeze(0).to(device)
    frame_queue.append(tensor)

    if len(frame_queue) > num_segments:
        frame_queue.pop(0)
    if len(frame_queue) < num_segments:
        cv2.imshow("🖐️ Gesture Recognition", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    clip = torch.cat(frame_queue, dim=0).unsqueeze(0)  # (1, T, C, H, W)
    clip = clip.view(1, num_segments, 3, 224, 224)
    with torch.no_grad():
        out = model(clip)
        probs = F.softmax(out, dim=1)[0]
        ema = probs if ema is None else 0.8 * ema + 0.2 * probs
        conf, idx = ema.max(0)
        label = labels[idx]

    now = time.time()
    fps = 1.0 / (now - prev)
    prev = now

    cv2.putText(frame, f"{label}", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
    cv2.putText(frame, f"{conf:.2f}", (30, 110), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)
    cv2.putText(frame, f"{fps:.1f} FPS", (30, 160), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)

    cv2.imshow("🖐️ Gesture Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



                TSN Configurations:
                    base model:         mobilenetv3_deptheca_mega
                    num_segments:       8
                    dropout_ratio:      0.0
                    shift_div:          8
                
=> base model: mobilenetv3_deptheca_mega
Patched GatedDTSM(in_channels=24) into InvertedResidual(
  (block): Sequential(
    (0): GatedDTSM(
      (net): Conv2dNormActivation(
        (0): Conv2d(24, 88, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(88, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (dw_temporal): Conv3d(24, 24, kernel_size=(3, 1, 1), stride=(1, 1, 1), padding=(1, 0, 0), groups=24, bias=False)
      (bn3d): BatchNorm3d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (fc1): Linear(in_features=24, out_features=6, bias=False)
      (fc2): Linear(in_features=6, out_features=24, bias=True)
    )
    (1): Conv2dNormAc